# 12장 클린 아키텍처를 향한 여정: 다음 단계

파이썬으로 구현하는 클린 아키텍처 - 12장 클린 아키텍처를 향한 여정: 다음 단계 코드 예제

## 개요

탐구를 마무리하며, 이제 작업 관리 구현을 넘어 클린 아키텍처 원칙을 더 넓게 적용할 수 있는지 ﻿살펴보려 한다. 이 여정을 통해 클린 아키텍처가 유연하고 유지 보수 가능하며 변화에 강한 시스템을 만드는 방법을 확인했다.

이 장에서 다루는 주요 주제:
* 이전 장들에서 ﻿살펴본 클린 아키텍처: 전체 조감
* 시스템 타입에 따른 클린 아키텍처 적용
* 아키텍처 리더십과 커뮤니티 참여

### 00_create_task_req_orig_vs_api.py

## 원본 vs API 요청 모델

작업 관리 시스템과 API 우선 시스템의 핵심 차이:
- **작업 관리 시스템**: 모델이 프레젠테이션 계층(CLI/웹 UI) 뒤에 숨겨져 자유롭게 변경 가능
- **API 우선 시스템**: 모델이 공개 계약으로 직접 노출되어 호환성 유지 필요

In [ ]:
# 원본(내부 전용) vs API(공개 계약) 요청 모델 비교
# 내부 전용: 프레젠테이션 계층 뒤에 숨겨져 있어 자유롭게 변경 가능
# API 공개 계약: 외부 소비자가 직접 사용하므로 호환성 유지 필요
# Colab에서도 외부 패키지 없이 실행 가능 (Python 표준 라이브러리만 사용)
from typing import Optional  # [추가] Optional 임포트
from uuid import UUID  # [추가] UUID 임포트

# 작업 관리 요청 모델 - 내부 전용 (CLI/웹 UI 뒤에 숨겨짐)
class CreateTaskRequest:
    """내부 전용 요청 모델 - to_execution_params()로 복잡한 변환 수행"""

    title: str
    description: str
    project_id: Optional[str] = None

    def to_execution_params(self) -> dict:
        """UI 형식 → 도메인 형식 변환 (문자열 정리, UUID 변환 등)"""
        return {
            "title": self.title.strip(),
            "project_id": UUID(self.project_id) if self.project_id else None,
            "description": self.description.strip(),
        }


# API 요청 DTO - 공개 계약 (외부 클라이언트가 직접 사용)
# 변환 메서드 불필요 - 클라이언트가 이미 구조화된 데이터를 전송
class CreateTaskRequest:
    title: str
    description: str
    project_id: Optional[str] = None

### 01_create_task_req_validation.py

## 요청 유효성 검증

API 시스템에서는 UI 관련 변환(`to_execution_params`)이 불필요하며, 클라이언트가 이미 구조화된 데이터를 전송하므로 요청 모델은 **유효한 입력 구조 정의**에만 집중한다. 클린 아키텍처의 인터페이스 어댑터 계층(컨트롤러)이 API 계약 안정성과 도메인 격리를 동시에 보장한다.

In [ ]:
# 수동 검증 vs Pydantic 자동 검증 비교
# 작업 관리(수동): __post_init__에서 직접 검증 코드 작성
# API(Pydantic): Field(..., min_length=1) 선언으로 자동 검증
# Colab에서 Pydantic 사용 시: !pip install pydantic 실행 필요

# 방식 1: 작업 관리 시스템 - 수동 검증 (dataclass + __post_init__)
class CreateTaskRequest:
    """수동 검증 방식 - 직접 검증 코드 작성 필요"""
    title: str
    description: str

    def __post_init__(self):
        # 수동 검증: 빈 제목 거부
        if not self.title.strip():
            raise ValueError("제목은 비어 있을 수 없습니다")

    def to_execution_params(self) -> dict:
        """UI 형식 → 도메인 형식 변환"""
        return {"title": self.title.strip(), "description": self.description.strip()}


# 방식 2: FastAPI/Pydantic - 선언적 자동 검증
from pydantic import BaseModel, Field


class CreateTaskRequest(BaseModel):
    """Pydantic 자동 검증 - 필드 제약 조건으로 검증 자동 처리"""
    title: str = Field(..., min_length=1)  # min_length=1로 빈 문자열 자동 거부
    description: str

### 02_create_task_route.py

## 작업 생성 라우트

Pydantic `BaseModel`을 확장한 `CreateTaskRequest`로 `min_length=1` 같은 필드 제약 조건을 선언하여 검증을 자동화한다.

In [ ]:
# FastAPI에서의 자동 검증: 라우트 핸들러 실행 전에 Pydantic이 검증 수행
# 유효하지 않은 요청 → 422 Unprocessable Entity 자동 응답
# 수동 검증 코드 불필요 → 보일러플레이트 대폭 감소
# Colab에서 FastAPI 사용 시: !pip install fastapi 실행 필요
# [수정] fastapi 미설치 시에도 동작하도록 보호
try:
    from fastapi import FastAPI
except ImportError:
    class FastAPI:  # [추가] 스텁 - Colab에서 fastapi 미설치 시 코드 구조 학습용
        def __init__(self, **kw): pass
        def post(self, *a, **kw):
            def decorator(f): return f
            return decorator
        def get(self, *a, **kw):
            def decorator(f): return f
            return decorator

app = FastAPI()


# [추가] task_controller 스텁 - 노트북/Colab 시연용
class _StubResult:
    success = {"message": "작업 생성됨"}
    is_success = True  # [추가] is_success 속성
    error = None
class _StubController:
    def handle_create(self, *args, **kwargs): return _StubResult()
task_controller = _StubController()


# FastAPI/Pydantic 자동 검증 라우트
@app.post("/tasks/")
def create_task(task_data: CreateTaskRequest):
    # FastAPI가 Pydantic 모델로 모든 필드를 자동 검증
    # 유효하지 않은 요청은 422 Unprocessable Entity로 자동 거부

    # 검증 통과 후 컨트롤러에 위임
    result = task_controller.handle_create(title=task_data.title, description=task_data.description)
    return result.success


# 클라이언트가 빈 제목(유효하지 않은 데이터)을 보낸 경우:

"""
{
  "title": "",
  "description": "테스트 기술"
}
"""

# FastAPI가 자동으로 검증 오류 응답을 반환:

"""
{
  "detail": [
    {
      "loc": ["body", "title"],
      "msg": "ensure this value has at least 1 characters",
      "type": "value_error.any_str.min_length",
      "ctx": {"limit_value": 1}
    }
  ]
}
"""

### 03_clean_arch_approach.py

## 클린 아키텍처 접근법: 순수성 vs 실용성

Pydantic 모델이 내부 계층으로 침투하지 않도록, 라우트 핸들러에서 프레임워크 모델 → 내부 도메인 모델로 명시적 변환을 수행하는 순수 접근법이다.

In [ ]:
# 순수 클린 아키텍처 접근법: Pydantic 모델의 내부 계층 침투 방지
# 프레임워크(FastAPI/Pydantic) 계층 → 내부 도메인 모델로 명시적 변환
# 순수성 vs 실용성의 트레이드오프: 변환 코드 추가 부담 vs 계층 독립성 확보
# Colab에서도 외부 패키지 없이 실행 가능 (위 셀의 스텁 활용)
from dataclasses import dataclass  # [추가] dataclass 임포트


# [추가] InternalCreateTaskRequest 스텁
# 내부 전용 요청 모델 - Pydantic에 의존하지 않는 순수 Python 데이터 클래스
@dataclass
class InternalCreateTaskRequest:
    title: str
    description: str


# FastAPI를 사용한 순수 클린 아키텍처 접근법
# 라우트 핸들러에서 Pydantic → 내부 모델 변환을 명시적으로 수행
@app.post("/tasks/")
def create_task(
    task_data: CreateTaskRequest,
):  # 여기서 Pydantic은 프레임워크 계층에 있음
    # Pydantic 모델이 내부 계층으로 침투하지 않도록
    # 프레임워크 모델 → 내부 도메인 모델로 변환 (계층 경계 보호)
    request = InternalCreateTaskRequest(
        title=task_data.title.strip(), description=task_data.description.strip()
    )

    # 내부 모델을 컨트롤러에 전달 - 컨트롤러는 Pydantic 존재를 알 필요 없음
    result = task_controller.handle_create(request)
    return result.success

### 04_fast_api_create_task.py

## FastAPI로 작업 생성

Flask에서 사용한 클린 아키텍처 패턴(컨트롤러, 요청/응답 DTO)을 FastAPI/Pydantic 환경에 적용한 예시이다.

In [ ]:
# FastAPI 라우트 핸들러: 프레임워크 계층(infrastructure/api/routes.py)에 해당
# 컨트롤러 패턴을 통한 API ↔ 도메인 간 변환 및 오류 처리
# response_model로 응답 직렬화 자동 처리, status_code=201로 생성 성공 표시
# Colab에서 FastAPI 사용 시: !pip install fastapi 실행 필요 (스텁으로도 구조 학습 가능)

# [수정] fastapi 미설치 시에도 동작하도록 보호
try:
    from fastapi import HTTPException
except ImportError:
    class HTTPException(Exception):  # [추가] 스텁 - Colab에서 fastapi 미설치 시 사용
        def __init__(self, status_code=400, detail=""): pass

try:
    from pydantic import BaseModel as _BaseModel
except ImportError:
    _BaseModel = type  # [추가] 스텁 - Colab에서 pydantic 미설치 시 사용


# [추가] TaskResponse 스텁 - API 응답 모델 (Pydantic 기반 자동 직렬화)
class TaskResponse(_BaseModel):
    message: str = "작업 생성됨"


# 프레임워크 계층 (infrastructure/api/routes.py)
# Flask 앱에서 사용한 것과 동일한 클린 아키텍처 패턴을 FastAPI에 적용
@app.post("/tasks/", response_model=TaskResponse, status_code=201)
def create_task(task_data: CreateTaskRequest):
    """새 작업 생성 - FastAPI가 Pydantic으로 요청 자동 검증"""
    # 컨트롤러가 API ↔ 도메인 간 변환을 처리 (인터페이스 어댑터 역할)
    result = task_controller.handle_create(
        title=task_data.title,
        description=task_data.description,
        project_id=task_data.project_id
    )

    if not result.is_success:
        # 프레임워크 경계에서 도메인 오류 → HTTP 오류로 변환
        raise HTTPException(status_code=400, detail=result.error.message)

    return result.success  # TaskResponse로 자동 직렬화 (Pydantic 처리)

### 05_task_entity_anti_pattern.py

## 안티 패턴: 엔터티의 프레임워크 의존성

도메인 엔티티가 인프라(Kafka)에 직접 의존하는 안티패턴이다. 도메인 이벤트를 일급 시민으로 끌어올려 이 문제를 해결한다.

In [ ]:
# 안티패턴: 도메인 엔티티가 인프라(Kafka)에 직접 의존
# 클린 아키텍처 위반 - 내부 원(도메인)이 외부 원(인프라)을 직접 참조
# 문제점: 테스트 시 Kafka 서버 필요, 메시징 시스템 교체 시 엔티티 수정 필요
# Colab에서는 kafka-python 미설치 시 스텁으로 동작 (구조 학습에 지장 없음)
from uuid import UUID  # [추가] UUID 임포트
from datetime import datetime  # [추가] datetime 임포트
from enum import Enum  # [추가] Enum 임포트
import json  # [추가] json 임포트

try:
    from kafka import KafkaProducer  # [수정] 외부 패키지 - Colab: !pip install kafka-python
except ImportError:
    class KafkaProducer:  # [추가] 스텁 - kafka 미설치 환경용
        def __init__(self, **kwargs): pass
        def send(self, topic, value): pass


# [추가] TaskStatus 스텁 - 작업 상태 열거형
class TaskStatus(Enum):
    TODO = "todo"
    IN_PROGRESS = "in_progress"
    DONE = "done"


# 안티패턴: 도메인 엔티티가 직접 이벤트를 발행
# 비즈니스 로직(상태 변경)과 인프라 관심사(메시징)가 혼합
class Task:
    def complete(self, user_id: UUID):
        # 도메인 로직: 작업 완료 상태 변경
        self.status = TaskStatus.DONE
        self.completed_at = datetime.now()
        self.completed_by = user_id

        # 안티패턴: 메시징 시스템에 대한 직접 의존성 → 클린 아키텍처 위반
        # Kafka 인프라 코드가 도메인 엔티티 안에 침투
        # → 테스트 시 Kafka 서버 필요, 메시징 교체 시 엔티티 수정 불가피
        kafka_producer = KafkaProducer(bootstrap_servers='kafka:9092')
        event_data = {
            "task_id": str(self.id),
            "completed_by": str(user_id),
            "completed_at": self.completed_at.isoformat()
        }
        kafka_producer.send('task_events', json.dumps(event_data).encode())

### 06_event_driven_task_create.py

## 이벤트 기반 아키텍처에서의 클린 아키텍처

올바른 해결: 도메인 엔티티는 순수 비즈니스 로직(상태 변경, 검증)만 담당하고, 이벤트 생성과 발행은 애플리케이션 계층(유스케이스)이 조율한다. 이를 통해 도메인이 인프라(Kafka 등)에 독립적이 되어 테스트 가능성과 유연성을 확보한다.

In [ ]:
# 이벤트 기반 클린 아키텍처: 안티패턴의 올바른 해결 방법
# 도메인 엔티티 → 순수 비즈니스 로직만 담당 (메시징 의존성 제거)
# 애플리케이션 계층(유스케이스) → 도메인 작업 + 이벤트 발행 조율
# 의존성 역전: EventPublisher 추상 인터페이스로 인프라 분리
# Colab에서도 외부 패키지 없이 실행 가능 (Python 표준 라이브러리만 사용)
from uuid import UUID  # [추가] UUID 임포트
from datetime import datetime  # [추가] datetime 임포트
from dataclasses import dataclass, field  # [추가] dataclass 임포트
from abc import ABC, abstractmethod  # [추가] ABC 임포트
from typing import Any  # [추가] Any 임포트


# [추가] TaskStatus 스텁 (이전 셀에서 정의되었지만 독립 실행을 위해 재정의)
from enum import Enum
class TaskStatus(Enum):
    TODO = "todo"
    IN_PROGRESS = "in_progress"
    DONE = "done"


# [추가] TaskRepository 스텁 - 리포지토리 추상 인터페이스 (포트)
class TaskRepository(ABC):
    @abstractmethod
    def get_by_id(self, task_id: UUID): pass
    @abstractmethod
    def save(self, task): pass


# [추가] EventPublisher 스텁 - 이벤트 발행 추상 인터페이스 (포트)
# 실제 구현체(Kafka, RabbitMQ 등)는 인프라 계층에서 제공
class EventPublisher(ABC):
    @abstractmethod
    def publish(self, event): pass


# [추가] Result 스텁 - 성공/실패를 명시적으로 표현하는 결과 객체
class Result:
    def __init__(self, value=None, error=None):
        self._value = value
        self._error = error
    @classmethod
    def success(cls, value): return cls(value=value)
    @classmethod
    def failure(cls, error): return cls(error=error)


# [추가] Error 스텁 - 도메인 오류 표현
class Error:
    def __init__(self, message: str):
        self.message = message


# [추가] TaskCompletedEvent 스텁 - 도메인 이벤트 (작업 완료 시 발생)
class TaskCompletedEvent:
    @classmethod
    def from_task(cls, task, user_id): return cls()


# 클린 도메인 엔티티 - 순수 비즈니스 로직만 담당
# 메시징(Kafka), 영속성(DB) 등 인프라 의존성 완전 제거
# → 단위 테스트 시 외부 시스템 불필요, 독립적 테스트 가능
class Task:
    def complete(self, user_id: UUID) -> None:
        # 비즈니스 규칙: 이미 완료된 작업 재완료 방지
        if self.status == TaskStatus.DONE:
            raise ValueError("이미 완료된 작업")
        # 상태 변경만 수행 - 이벤트 발행은 유스케이스 책임
        self.status = TaskStatus.DONE
        self.completed_at = datetime.now()
        self.completed_by = user_id


# 애플리케이션 계층: 도메인 작업과 이벤트 발행을 조율하는 유스케이스
# 의존성 주입으로 TaskRepository, EventPublisher 추상 인터페이스를 수신
# → Kafka든 RabbitMQ든 구현체 교체 시 유스케이스 코드 변경 불필요
@dataclass
class CompleteTaskUseCase:
    task_repository: TaskRepository    # 포트: 영속성 추상 인터페이스
    event_publisher: EventPublisher    # 포트: 이벤트 발행 추상 인터페이스

    def execute(self, task_id: UUID, user_id: UUID) -> Result:
        try:
            # 1. 리포지토리에서 작업 조회 (영속성 계층 추상화)
            task = self.task_repository.get_by_id(task_id)
            # 2. 도메인 로직 실행 (순수 비즈니스 규칙)
            task.complete(user_id)
            # 3. 변경된 작업 저장
            self.task_repository.save(task)

            # 4. 도메인 이벤트 생성 및 추상 인터페이스를 통해 발행
            # 실제 메시징 시스템(Kafka 등)은 인프라 계층의 EventPublisher 구현체가 처리
            event = TaskCompletedEvent.from_task(task, user_id)
            self.event_publisher.publish(event)

            return Result.success(task)
        except ValueError as e:
            # 도메인 규칙 위반 시 실패 결과 반환
            return Result.failure(Error(str(e)))